In [32]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
from torch import Tensor

import math
import random


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "attention" / "__init__.py").exists() and (
            candidate / "date2date" / "data.py"
        ).exists():
            return candidate
    raise RuntimeError(f"找不到项目根目录，起始路径: {start}")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from attention import MultiHeadAttention


embeding时，我们将embeding分量初始化为均值为0，标准差为 (d_model)^(-1/2)。每一个分量都满足均值为0，标准差为(d_model)^(-1/2)的正太分布，每一个embedding向量的范数是1。
E(||x||^2)=\sum_{i=1}^{d_model}E{(x_i)^2}
E((x_i)^2) = E(((x_i)-E(x))^2) = Var(x), 因为 E(x_i) = 0 和 Var(x_i)= d_model^(-1)，所以E((x_i)^2) = d_model^(-1)。
所以E(||x||^2)=1


In [33]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, scale: bool = True) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        # 放大 sqrt(d_model)，让 token 信号与位置编码处于同一量级
        self.scale = scale
        # 可训练参数：每行是一个 token 的向量
        self.weight = nn.Parameter(torch.empty(vocab_size, d_model))
        nn.init.normal_(self.weight, mean=0.0, std=d_model ** -0.5)

    def forward(self, token_ids: Tensor) -> Tensor:
        """
        token_ids: (B, L), dtype=torch.long
        返回: (B, L, D)
        """
        # (B, L) -> (B, L, D)
        x = self.weight[token_ids]          # 花式索引即查表
        if self.scale:
            x = x * math.sqrt(self.d_model)
        return x
        
    

位置编码
我们将每一个token的每两个特征设置为一组，设token是整个序列中的第i个token，该token的第j组。
则有：p_{i,2j} = sin(\frac{i}{10000^{2j/d}})
p_{i,2j+1} = cos(\frac{i}{10000^{2j/d}})
\(i\) 决定当前采样位置，\(j\) 决定使用哪一种频率。固定 \(j\) 沿 \(i\) 观察时频率不变；切换 \(j\) 时，频率才会改变。

In [34]:
class PositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model) -> None:
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        # 构造分母
        index = torch.arange(0, d_model, 2).float()
        scale = -math.log(10000.0) / d_model
        div_term = torch.exp(index * scale)
        position = torch.arange(max_len).unsqueeze(1)
        pe[:,0::2] = torch.sin(div_term * position)
        pe[:,1::2] = torch.cos(div_term * position)
        self.register_buffer("pe",pe.unsqueeze(0))

    def forward(self, x: Tensor, start_pos: int = 0) -> Tensor:
        L = x.shape[1]
        max_len = self.pe.shape[1]
        if start_pos < 0 or start_pos + L > max_len:
            raise ValueError(
                f"位置越界：start_pos={start_pos}, L={L}, max_len={max_len}"
            )
        return x + self.pe[:,start_pos:start_pos+L,:]
        

    
    

多层感知机（MLP）：由多个全连接层堆叠而成，每层与上一层完全连接。核心是非线性激活函数——没有它多层会退化为单层线性变换。常见激活函数（ReLU 等）是固定函数，没有可训练参数；可训练的是全连接层的权重和偏置。

In [35]:
class PositionWiseFFN(nn.Module):
    def __init__(self, d_model, d_ff) -> None:
        super().__init__()
        self.dense1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.dense2 = nn.Linear(d_ff, d_model)

    def forward(self, x: Tensor) -> Tensor:
        return self.dense2(self.relu(self.dense1(x)))

标准化
减去均值：得到相对“中心位置”的偏移，消除了整体平移。
除以标准差：把偏移量按该组数据自身的波动尺度来衡量；结果表示“离均值多少个标准差”，而不是一般意义上的比例。
标准化通常是
\[
z=\frac{x-\mu}{\sigma}
\]这样处理后，数据均值约为 0、标准差约为 1，方便不同量纲、不同数值范围的特征被模型以更接近的尺度处理。

LayerNorm和batchNorm
layerNorm在同一个token中对所有特征进行标准化。果某个特征在这个 token 里是异常大的 outlier，会把其他特征一起压扁。特征维度少或特征含义差异大时，LN 的效果受影响
batchNorm对同一个Batch中对所有token的同一个特征进行标准化。特征之间互不干扰，保留了每个通道独立的分布形状

In [36]:
ln = nn.LayerNorm(2)
bn = nn.BatchNorm1d(2)
X = torch.tensor([[1, 2], [2, 3]], dtype=torch.float32)
# 在训练模式下计算X的均值和方差
print('layer norm:', ln(X), '\nbatch norm:', bn(X))

layer norm: tensor([[-1.0000,  1.0000],
        [-1.0000,  1.0000]], grad_fn=<NativeLayerNormBackward0>) 
batch norm: tensor([[-1.0000, -1.0000],
        [ 1.0000,  1.0000]], grad_fn=<NativeBatchNormBackward0>)


残差网络
从y = F(x)变成 y = x + F(x) 时，实际上神经网络学习到的是从F(x)如何得到target y,到F(x)如何得到 target y -x 。如果神经网络什么也没学到，也就是 target y -x  = 0，那至少在y = x + F(x) 这个式子中，x还能继续向下传。

Pre-LN和Post-LN
Pre-LN: y = x + f(layerNorm(x))
Post-LN: y = layerNorm(x + f(x))

Pre-LN与Post-LN相比，x传输的更直接，更原始，也就更容易保持训练稳定。

In [37]:
class AddNorm(nn.Module):
    # layernomr需要先初始化才能用
    def __init__(self, d_model, dropout) -> None:
        super().__init__()
        self.ln = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X, Y) -> Tensor:
        return self.ln(X + self.dropout(Y))


自注意力
以原始的token embedding为基础，经过Wq，Wk，Wv的投影生成query，key，value向量，在一个seq内部，对seq的每个token都生成query，key，value向量，在使用注意力机制进行评分

Padding Mask 
让注意力忽视pad token。padding mask直接检查seq中的token id，从而生成mask。pad token也会进行embedding和positional。
mask一定是在soft max之前做。因为放入soft max后，分数就会生成score了。padding的分数是不能进score的。
padding mask会遮挡Q·Kᵀ 的列，相当于阻止padding作为key被比较。
        key0   key1   key_pad
query0   .      .      -inf
query1   .      .      -inf
query2   .      .      -inf

解码器
解码器与编码器的差别是解码器的掩码注意力
编码器-解码器注意力也是注意力，只是query，key，value不再是同一个向量

Causal Mask / Look-ahead Mask
第i 个query只能看见 j<=i的key

In [38]:
class DecoderBlock(nn.Module):
    def __init__(self, query_size, key_size, value_size, head_num, d_model, dropout, d_ff) -> None:
        super().__init__()
        # 初始化中，初始化的是结构，所以编码器-解码器注意力和解码器注意力一样
        # 解码器注意力
        self.attention_layer1 = MultiHeadAttention(query_size, key_size, value_size, d_model, head_num)
        self.add_norm1 = AddNorm(d_model, dropout)

        # 前馈网络
        self.ffn_layer = PositionWiseFFN(d_model, d_ff)
        self.add_norm3 = AddNorm(d_model, dropout)

    def forward(self, decoder_output, encoder_valid_lens: torch.Tensor | None = None, decoder_valid_lens: torch.Tensor | None = None) -> Tensor:
        # attention1是decoder自注意力，使用casual mask
        tensor1 = self.add_norm1(decoder_output, self.attention_layer1(decoder_output, decoder_output, decoder_output, decoder_valid_lens, True))
        return self.add_norm3(tensor1, self.ffn_layer(tensor1))

In [39]:
class TransformerDecoder(nn.Module):
    # layer_num是decoderBlock的个数
    def __init__(self, vocab_size, max_len, d_model, layer_num, query_size, key_size, value_size, head_num, dropout, d_ff) -> None:
        super().__init__()
        self.tokenEmbedding = TokenEmbedding(vocab_size, d_model)
        self.positionalEncoding = PositionalEncoding(max_len, d_model)
        self.blks = nn.Sequential()
        for i in range(layer_num):
            self.blks.add_module("block"+str(i),DecoderBlock(query_size, key_size, value_size, head_num, d_model, dropout, d_ff))
        self.dense = nn.Linear( d_model, vocab_size)
        

    #seqs 的形状(B,L) L表示有几个序列
    def forward(self, decoder_seqs, decoder_valid_lens: torch.Tensor | None = None) -> Tensor:
        x = self.positionalEncoding(self.tokenEmbedding(decoder_seqs))
        for i, blk in enumerate(self.blks):
            x = blk(x, decoder_valid_lens)

        return self.dense(x)
        



In [40]:
class OnlyDecoder(nn.Module):
    def __init__(self, vocab_size, max_len, d_model, layer_num, query_size, key_size, value_size, head_num, dropout, d_ff):
        super().__init__()
        self.decoder = TransformerDecoder(vocab_size, max_len, d_model, layer_num, query_size, key_size, value_size, head_num, dropout, d_ff)
    def forward(self, decoder_seqs, decoder_valid_lens)-> Tensor:
        decoder_output = self.decoder(decoder_seqs, decoder_valid_lens)
        return decoder_output

In [41]:
from date2date.data import DEVICE, fixed_samples, vocab_size, tensor_from_string
from date2date.train import  _validate_string_batch
from date2date.experiment_log import (
    PROJECT_ROOT,
    RUNS_DIR,
    append_index,
    git_commit,
    load_or_create_fixed_test,
    make_training_samples,
    new_run_id,
    save_checkpoint,
    write_json,
)
from date2date.report import update_readme
from date2date.train import char_accuracy, eval_samples, generate

In [42]:
def train_batch_samples(
    input_strs,
    target_strs,
    model: nn.Module,
    model_optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
):
    input_batch = _validate_string_batch(input_strs, "input_strs")
    target_batch = _validate_string_batch(target_strs, "target_strs")
    if len(input_batch) != len(target_batch):
        raise ValueError("input_strs and target_strs must contain the same number of samples")

    model.train()
    encoder_input = torch.cat(
        [tensor_from_string(s) for s in input_batch],
        dim=0
    )
    # decoder_input.shape = batch,seq
    decoder_input  = torch.cat(
        [tensor_from_string(s, add_eos=True)  for s in target_batch],
        dim=0)  # <SOS>abc
    # target_tensor.shape = batch,seq
    target_tensor  = torch.cat([tensor_from_string(s, add_eos=True)  for s in target_batch], 
        dim=0)  # abc<EOS>

    # encoder_input: (B, 10)   decoder_input: (B, 11)
    sep_col = torch.full(
        (encoder_input.size(0), 1),
        SEP_token,
        dtype=torch.long,
        device=encoder_input.device,
    )

    # (B, 10) + (B, 1) + (B, 11) -> (B, 22)
    only_decoder_input = torch.cat([encoder_input, sep_col, decoder_input], dim=1)
    model.zero_grad()
    model_optimizer.zero_grad()

    # model_outputs.shape = batch,seq,vocab_size
    model_outputs = model(only_decoder_input, None)
    model_outputs = model_outputs[:,:-1,:]
    labels = only_decoder_input[:,1:]

    # 先在三维上切掉前 10 个位置（源 0..9 + SEP 不需要算 loss）
    logits = model_outputs[:, 10:, :].reshape(-1, vocab_size)   # (B*11, V)
    target = labels[:, 10:].reshape(-1)                         # (B*11,)
    loss = criterion(logits, target)
    loss.backward()
    model_optimizer.step()
    return loss.item()

In [43]:
def train_epochs(
    train_samples: list[tuple[str, str]],
    model: nn.Module,
    model_optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    epochs: int,
    batch_size: int,
    seed: int,
) -> list[float]:
    epoch_losses = []
    shuffle_rng = random.Random(seed)
    for epoch in range(epochs):
        shuffle_rng.shuffle(train_samples)
        epoch_loss = 0.0
        for start in range(0, len(train_samples), batch_size):
            batch_samples = train_samples[start : start + batch_size]
            input_strs, target_strs = zip(*batch_samples)
            batch_loss = train_batch_samples(
                input_strs,
                target_strs,
                model,
                model_optimizer,
                criterion,
            )
            epoch_loss += batch_loss * len(batch_samples)
        avg_loss = epoch_loss / len(train_samples)
        epoch_losses.append(avg_loss)
        print(f"epoch {epoch + 1}/{epochs} avg_loss={avg_loss:.6f}")
    return epoch_losses

In [44]:
from date2date.data import EOS_token, SOS_token, SEP_token, string_from_tensor
max_new_tokens = 22

In [45]:
max_len = 22
d_model = 16
layer_num = 2
query_size = key_size = value_size = d_model
head_num = 4
dropout = 0.1
d_ff = 64
lr=1e-3

In [46]:
train_size = 1000
seed = 42
epochs = 40
batch_size = 100
test_seed = 20260710
test_size = 100
test_samples = load_or_create_fixed_test(test_seed, test_size)

samples = make_training_samples(train_size, seed, {input_str for input_str, _ in test_samples},)

model = OnlyDecoder(vocab_size, max_len, d_model, layer_num, query_size, key_size, value_size, head_num, dropout, d_ff)
model_optimizer = torch.optim.Adam(model.parameters(), lr)
criterion = nn.CrossEntropyLoss()

epoch_losses = train_epochs(samples, model, model_optimizer, criterion, epochs, batch_size, seed)

epoch 1/40 avg_loss=2.637913
epoch 2/40 avg_loss=2.415787
epoch 3/40 avg_loss=2.277942
epoch 4/40 avg_loss=2.165731
epoch 5/40 avg_loss=2.061217
epoch 6/40 avg_loss=1.952974
epoch 7/40 avg_loss=1.847503
epoch 8/40 avg_loss=1.747722
epoch 9/40 avg_loss=1.658814
epoch 10/40 avg_loss=1.577195
epoch 11/40 avg_loss=1.506932
epoch 12/40 avg_loss=1.448427
epoch 13/40 avg_loss=1.391920
epoch 14/40 avg_loss=1.332310
epoch 15/40 avg_loss=1.278228
epoch 16/40 avg_loss=1.221736
epoch 17/40 avg_loss=1.168039
epoch 18/40 avg_loss=1.117649
epoch 19/40 avg_loss=1.067709
epoch 20/40 avg_loss=1.016546
epoch 21/40 avg_loss=0.966512
epoch 22/40 avg_loss=0.920646
epoch 23/40 avg_loss=0.879613
epoch 24/40 avg_loss=0.843108
epoch 25/40 avg_loss=0.805355
epoch 26/40 avg_loss=0.771692
epoch 27/40 avg_loss=0.739146
epoch 28/40 avg_loss=0.703293
epoch 29/40 avg_loss=0.667568
epoch 30/40 avg_loss=0.634813
epoch 31/40 avg_loss=0.589749
epoch 32/40 avg_loss=0.536116
epoch 33/40 avg_loss=0.486865
epoch 34/40 avg_los

推理

In [48]:
def generate(input_str, model:nn.Module, max_new_tokens: int = 20):
    model.eval()
    with torch.no_grad():

        prompt = torch.cat([tensor_from_string(input_str)], torch.tensor([[SEP_token]], dtype=torch.long, device=DEVICE),
            dim=1,
        )
        outputs = []
        
        for _ in range(max_new_tokens):
            outputs = model(prompt, None)          # (1, L, vocab_size)
            pred_token_id = outputs[0, -1].argmax(dim=-1)
            if pred_token_id.item() == EOS_token:
                break
            prompt = torch.cat([prompt, pred_token_id.view(1, 1)], dim=1)
        if not outputs:
            return ""


    return string_from_tensor(torch.stack(outputs).unsqueeze(0))

In [49]:
def eval_samples(samples, model:nn.Module, verbose=False):
    model.eval()

    correct_count = 0

    for input_str, target_str in samples:
        pred = generate(input_str, model, max_new_tokens=max_new_tokens)
        correct = pred == target_str

        if correct:
            correct_count += 1

        if verbose:
            print("input:  ", input_str)
            print("pred:   ", pred)
            print("target: ", target_str)
            print("correct:", correct)
            print("---")

    total = len(samples)
    acc = correct_count / total

    return correct_count, total, acc

In [50]:
eval_samples(test_samples, model, True)

TypeError: cat() received an invalid combination of arguments - got (list, Tensor, dim=int), but expected one of:
 * (tuple of Tensors tensors, int dim = 0, *, Tensor out = None)
 * (tuple of Tensors tensors, name dim, *, Tensor out = None)
